In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Mode Skim Matrices on the V2 Areas and the Flow Comparison

Step 26 (`GC_data_inventory_and_skims.ipynb`) filled the generalized-cost components mode by mode and left every cell with a status. This step turns them into **one complete 25 × 25 skim set per mode** — car, bus, Metronit (BRT), LRT all underground, LRT all ground — and then **compares the 2022 flows with the skims**: how the observed car / transit split responds to the cost difference, and what the two LRT scenarios would draw, pair by pair and link by link along the trunk.

**Skims.** Each mode gets an in-vehicle-time, walk, wait, transfer and partial generalized-cost matrix (`GC = IVT + 2·walk + 2·wait + 8·transfers`; money is out of the comparison by decision — the transit fare in the area is flat and integrated with a daily cap, the same for bus, Metronit and LRT and for every pair, so it drops out of the transit choice and is a constant per trip against the car), with a status matrix beside it. Two extensions over step 26:

- **LRT beyond the ten station areas.** `hf_lrt_3` serves areas 201–210 (Tirat Carmel – Hamifrats). For the other fifteen areas the LRT is reached by a **feeder bus leg** to the best gateway station area: the observed bus skim of step 30 (in-vehicle + wait + access walk), an 8-minute transfer penalty, then the LRT. Both ends off the line means two feeder legs. The gateway is the one that minimises the composite cost, and is recorded per pair.
- **Metronit** stays a direct-service skim (278 pairs); elsewhere the mode is *not available* rather than missing.

Intra-area cells (the diagonal) are not skimmed for the transit modes at this aggregation — a 25-area cell is too coarse for an intra-area transit trip — and are excluded from the flow comparison.

**Flow comparison** (`docs/LRT_CAPTURE_PLAN.md` §3). (i) The observed 2022 transit share per area pair against `GC_bus − GC_car`, fitted by a volume-weighted binary logit — the cost sensitivity λ. (ii) A nested incremental logit pivoted on the observed shares: bus → LRT within the transit nest at `λ_T`, car → transit through the improved transit logsum. (iii) The resulting LRT trips per pair, by source (from bus, from car / taxi), and their **loads on the trunk links** against today's bus potential movements on the same links. All flows are 06:00–09:00 trips with both ends in the 25 areas; peak-hour loads use the step 27 factors.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/skims'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)
W_WALK, W_WAIT, TRANSFER_PEN = 2.0, 2.0, 8.0
HEADWAY_LRT = 5.0                                    # min, 12 departures an hour (assumption of 22 Sep 2026)
WALK_SPEED, DETOUR = 80.0, 1.3
LRT_SCEN = ['lrt_all_underground', 'lrt_all_ground', 'lrt_design_50kmh']
MODES = ['car', 'bus', 'brt'] + LRT_SCEN
LABEL = {'car': 'Car', 'bus': 'Bus', 'brt': 'Metronit (BRT)', 'lrt_all_underground': 'LRT all underground', 'lrt_all_ground': 'LRT all ground', 'lrt_design_50kmh': 'LRT design 50 km/h'}
xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); areas = xl.parse('AreaCodes').set_index('AggCode'); key = xl.parse('TazAgg')
AREAS = list(areas.index); names = areas['AggAreaName']; A = len(AREAS)
TRUNK_ORDER = areas['Order_T1'][areas['Order_T1'].between(1, 12)].sort_values(); TRUNK = list(TRUNK_ORDER.index)   # 201 … 212
def mat(v=np.nan): return pd.DataFrame(v, index=AREAS, columns=AREAS, dtype=float)
def smat(v='missing'): return pd.DataFrame(v, index=AREAS, columns=AREAS, dtype=object)
def set_diag(df, v):
    a = df.to_numpy(copy=True); np.fill_diagonal(a, v); return pd.DataFrame(a, index=df.index, columns=df.columns)
# ---- step 26 components, long -> matrices ----
comp = pd.read_csv('Output/gc/gc_components_area_v2_long.csv')
def pick(mode, component, col='value'):
    s = comp[(comp['mode'] == mode) & (comp['component'] == component)].pivot(index='o', columns='d', values=col).reindex(index=AREAS, columns=AREAS)
    return s.astype(float) if col == 'value' else s.fillna('missing')
C = {(m, c): pick(m, c) for m, c in comp.groupby(['mode', 'component']).size().index}
ST = {(m, c): pick(m, c, 'status') for m, c in C}
print(f"step 26 components loaded: {len(C)} mode × component matrices; trunk areas on hf_lrt_3: {[a for a in TRUNK if pd.notna(C[('lrt_all_underground', 'ivt')].loc[a, a])]}")

step 26 components loaded: 35 mode × component matrices; trunk areas on hf_lrt_3: [201, 202, 203, 204, 205, 206, 207, 208, 209, 210]


## 1. Car, bus and Metronit skims — as filled in step 26, with the diagonal rule made explicit

In [3]:
SK = {}   # SK[mode][component] -> matrix; SK[mode]['status'] -> status of the GC cell
def gc_of(ivt, walk, wait, tr): return ivt + W_WALK * walk + W_WAIT * wait + TRANSFER_PEN * tr
# car: door-to-door survey time; walk / wait / transfers are zero by construction
SK['car'] = {'ivt': C[('car', 'ivt')], 'walk': mat(0.0), 'wait': mat(0.0), 'transfers': mat(0.0), 'status': ST[('car', 'ivt')]}
# bus: observed in-vehicle time (step 30) on the direct pairs, scaled fastest-path floor elsewhere (assumed), scheduled kept for reference
bus_ivt, bus_walk, bus_wait = C[('bus', 'ivt')].copy(), C[('bus', 'walk')].copy(), C[('bus', 'wait')].copy()
bus_tr = C[('bus', 'transfers')].copy()
direct = ST[('bus', 'transfers')] == 'derived'
bus_tr = bus_tr.where(direct, 1.0)                     # no direct service: the floor is a no-transfer lower bound, so count one transfer (assumed)
bus_status = set_diag(smat('derived').where(direct, 'assumed'), 'not_skimmed')
bus_ivt, bus_walk, bus_wait, bus_tr = (set_diag(m_, np.nan) for m_ in (bus_ivt, bus_walk, bus_wait, bus_tr))
SK['bus'] = {'ivt': bus_ivt, 'ivt_scheduled': C[('bus', 'ivt_scheduled')], 'walk': bus_walk, 'wait': bus_wait, 'transfers': bus_tr, 'status': bus_status}
# Metronit: direct pairs only; elsewhere not available
brt_av = ST[('brt', 'ivt')] == 'derived'
SK['brt'] = {'ivt': C[('brt', 'ivt')], 'walk': C[('brt', 'walk')], 'wait': C[('brt', 'wait')], 'transfers': C[('brt', 'transfers')],
             'status': set_diag(smat('derived').where(brt_av, 'not_available'), 'not_skimmed')}
for m in ('car', 'bus', 'brt'):
    SK[m]['gc'] = gc_of(SK[m]['ivt'], SK[m]['walk'], SK[m]['wait'], SK[m]['transfers'])
    print(f"{LABEL[m]:15s} GC cells filled {int(SK[m]['gc'].notna().values.sum()):3d} of {A * A}; status: " + ', '.join(f"{k} {v}" for k, v in SK[m]['status'].stack().value_counts().items()))

Car             GC cells filled 625 of 625; status: derived 590, measured 35
Bus             GC cells filled 598 of 625; status: derived 422, assumed 178, not_skimmed 25
Metronit (BRT)  GC cells filled 278 of 625; status: not_available 322, derived 278, not_skimmed 25


## 2. LRT skims — direct on the line, by feeder bus from the other areas

For a pair with both ends among the ten station areas the LRT skim is step 26's: in-vehicle time of step 25, walk to the nearest station at each end (population-weighted access, employment-weighted egress), a 2.5-minute wait, no transfer.

For an origin off the line the traveller takes a feeder service to a **gateway** station area *g*, transfers, and rides the LRT. The feeder is either a **bus** (the observed bus skim of step 30; an 8-minute transfer penalty at the station) or, where a direct **Metronit** service reaches the gateway, the **Metronit** (its own observed skim; the LRT–Metronit transfer is **free by assumption** — same platform, integrated operation — so no penalty), whichever is cheaper. The composite generalized cost with a bus feeder is

`GC = [bus_ivt(o,g) + 2·bus_wait(o,g) + 2·bus_access_walk(o)] + 8 + [lrt_ivt(g,d) + 2·2.5 + 2·lrt_egress_walk(d)]`

with the feeder's egress walk and the LRT access walk at *g* dropped (the feeder stops at the station). Off-line destinations mirror this with an egress leg from a gateway *h*; both ends off the line combine the two legs. The gateway pair and the feeder modes minimising the composite cost are chosen per pair and recorded (`legs`: `directLRT`, `bus→LRT`, `brt→LRT`, `LRT→bus`, …). The `transfers` matrix counts the *penalised* transfers (bus ↔ LRT); Metronit ↔ LRT transfers are free and not counted, and LRT ↔ LRT transfers would be free too — none arise while `hf_lrt_3` is a single line, so the parameter waits for the branch alignments. Where a bus feeder leg rests on step 26's assumed floor (no direct bus service to the gateway), the LRT cell is flagged `assumed`.

In [4]:
# per-area bus walk (access population-weighted, egress employment-weighted), as in step 26, from the step 29 LOS table
los = pd.read_csv('Output/gtfs/bus_los_taz.csv').set_index('TAZ')
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='windows-1255').set_index('TAZ_ID')
tk = key.copy(); tk['walk_min'] = los['bus_nearest_stop_m'].reindex(tk['TAZ']).values * DETOUR / WALK_SPEED
tk['pop'] = zon['POPULATION'].fillna(0).reindex(tk['TAZ']).values; tk['emp'] = zon['EMPL_TOT'].fillna(0).reindex(tk['TAZ']).values
def wavg(g, col, w): return np.average(g[col], weights=g[w]) if g[w].sum() > 0 else g[col].mean()
accb = tk.groupby('AggCode').apply(lambda g: pd.Series({'access': wavg(g.dropna(subset=['walk_min']), 'walk_min', 'pop'), 'egress': wavg(g.dropna(subset=['walk_min']), 'walk_min', 'emp')}), include_groups=False).reindex(AREAS)
chk = (accb['access'].values[:, None] + accb['egress'].values[None, :]); assert np.allclose(chk, C[('bus', 'walk')].values, atol=0.02), 'bus walk per area does not reproduce step 26'
lrt_acc = pd.read_csv('Output/gc/lrt_access_area_v2.csv', index_col=0).reindex(AREAS)
ON_LINE = [a for a in AREAS if pd.notna(C[('lrt_all_underground', 'ivt')].loc[a, a])]
print(f"bus access walk per area {accb['access'].min():.1f}–{accb['access'].max():.1f} min; LRT station areas {ON_LINE}")
# feeder leg costs (origin side: to gateway; destination side: from gateway): bus (penalised transfer) or Metronit (free transfer), both directions
BRT_LRT_TRANSFER_PEN = 0.0                               # generalized minutes — LRT–Metronit transfers free by assumption (22 Sep 2026)
LRT_LRT_TRANSFER_PEN = 0.0                               # generalized minutes — LRT–LRT transfers free by the same assumption; unused while hf_lrt_3 is a single line (no LRT–LRT transfer arises), to be applied when the branch alignments exist
brt_ivt, brt_wait = SK['brt']['ivt'], SK['brt']['wait']
feed_assumed = ~direct
def feeder(x, g, side):
    """cheapest feeder leg between off-line area x and gateway g: (cost, ivt, wait, walk, penalised transfers, mode, assumed)"""
    o_, d_ = (x, g) if side == 'o' else (g, x); wk = accb.loc[x, 'access' if side == 'o' else 'egress']
    opts = []
    if pd.notna(bus_ivt.loc[o_, d_]):
        opts.append((bus_ivt.loc[o_, d_] + W_WAIT * bus_wait.loc[o_, d_] + W_WALK * wk + TRANSFER_PEN, bus_ivt.loc[o_, d_], bus_wait.loc[o_, d_], wk, 1.0, 'bus', bool(feed_assumed.loc[o_, d_])))
    if pd.notna(brt_ivt.loc[o_, d_]):
        opts.append((brt_ivt.loc[o_, d_] + W_WAIT * brt_wait.loc[o_, d_] + W_WALK * wk + BRT_LRT_TRANSFER_PEN, brt_ivt.loc[o_, d_], brt_wait.loc[o_, d_], wk, 0.0, 'brt', False))
    return min(opts, key=lambda t: t[0]) if opts else None
for sc in LRT_SCEN:
    lrt_ivt = C[(sc, 'ivt')]
    ivt, walk, wait, tr, gc, status = mat(), mat(), mat(), mat(), mat(), smat('missing')
    gw_o, gw_d, legs = smat(''), smat(''), smat('')
    for o in AREAS:
        for d in AREAS:
            if o == d: status.loc[o, d] = 'not_skimmed'; continue
            best = None
            G_o = [o] if o in ON_LINE else ON_LINE; G_d = [d] if d in ON_LINE else ON_LINE
            for g in G_o:
                for h in G_d:
                    if g == h: continue                                               # the LRT must be ridden between two distinct station areas
                    c_ivt = lrt_ivt.loc[g, h]; c_walk = 0.0; c_wait = HEADWAY_LRT / 2; c_tr = 0.0; asm = False; m_o = m_d = ''
                    if o in ON_LINE: c_walk += lrt_acc.loc[o, 'access_walk_min_pop']
                    else:
                        f = feeder(o, g, 'o')
                        if f is None: continue
                        c_ivt += f[1]; c_wait += f[2]; c_walk += f[3]; c_tr += f[4]; m_o = f[5]; asm |= f[6]
                    if d in ON_LINE: c_walk += lrt_acc.loc[d, 'egress_walk_min_emp']
                    else:
                        f = feeder(d, h, 'd')
                        if f is None: continue
                        c_ivt += f[1]; c_wait += f[2]; c_walk += f[3]; c_tr += f[4]; m_d = f[5]; asm |= f[6]
                    c_gc = c_ivt + W_WALK * c_walk + W_WAIT * c_wait + TRANSFER_PEN * c_tr
                    if best is None or c_gc < best[0]: best = (c_gc, c_ivt, c_walk, c_wait, c_tr, g, h, asm, m_o, m_d)
            if best is None: continue
            gc.loc[o, d], ivt.loc[o, d], walk.loc[o, d], wait.loc[o, d], tr.loc[o, d] = best[:5]
            gw_o.loc[o, d], gw_d.loc[o, d] = best[5], best[6]
            status.loc[o, d] = 'derived' if (o in ON_LINE and d in ON_LINE) else ('assumed' if best[7] else 'derived_feeder')
            legs.loc[o, d] = ('direct' if o in ON_LINE else best[8] + '→') + 'LRT' + ('' if d in ON_LINE else '→' + best[9])
    SK[sc] = {'ivt': ivt, 'walk': walk, 'wait': wait, 'transfers': tr, 'gc': gc, 'status': status, 'gateway_o': gw_o, 'gateway_d': gw_d, 'legs': legs}
    print(f"{LABEL[sc]:20s} GC cells {int(gc.notna().values.sum())} of {A * (A - 1)} off-diagonal; status: " + ', '.join(f"{k} {v}" for k, v in status.stack().value_counts().items()) + '; legs: ' + ', '.join(f"{k} {v}" for k, v in legs.stack().value_counts().items() if k))
# consistency with step 26 on the ten station areas
chk26 = pd.read_csv('Output/gc/gc_area_v2_lrt_all_underground.csv', index_col=0); chk26.columns = chk26.columns.astype(int)
dd = (SK['lrt_all_underground']['gc'] - chk26).loc[ON_LINE, ON_LINE].abs().values; dd = dd[~np.eye(len(ON_LINE), dtype=bool)]
assert np.nanmax(dd) < 0.05, 'LRT direct skim differs from step 26'
# the LRT in-vehicle part alone (without the feeder bus), for the profile table
for sc in LRT_SCEN:
    SK[sc]['ivt_lrt_only'] = pd.DataFrame([[C[(sc, 'ivt')].loc[SK[sc]['gateway_o'].loc[o, d], SK[sc]['gateway_d'].loc[o, d]] if SK[sc]['gateway_o'].loc[o, d] != '' else np.nan for d in AREAS] for o in AREAS], index=AREAS, columns=AREAS)

bus access walk per area 0.7–10.6 min; LRT station areas [201, 202, 203, 204, 205, 206, 207, 208, 209, 210]


LRT all underground  GC cells 600 of 600 off-diagonal; status: derived_feeder 435, derived 90, assumed 75, not_skimmed 25; legs: brt→LRT→brt 110, brt→LRT 102, directLRT 90, directLRT→brt 81, directLRT→bus 69, bus→LRT 48, brt→LRT→bus 44, bus→LRT→brt 41, bus→LRT→bus 15


LRT all ground       GC cells 600 of 600 off-diagonal; status: derived_feeder 391, assumed 119, derived 90, not_skimmed 25; legs: brt→LRT→brt 110, directLRT 90, brt→LRT 88, directLRT→brt 76, directLRT→bus 74, bus→LRT 62, brt→LRT→bus 44, bus→LRT→brt 38, bus→LRT→bus 18


LRT design 50 km/h   GC cells 600 of 600 off-diagonal; status: derived_feeder 468, derived 90, assumed 42, not_skimmed 25; legs: brt→LRT 109, brt→LRT→brt 100, directLRT 90, directLRT→brt 84, directLRT→bus 66, brt→LRT→bus 54, bus→LRT 41, bus→LRT→brt 38, bus→LRT→bus 18


## 3. The skim set — files, and a summary per mode on the trunk pairs

In [5]:
COMPS = ['ivt', 'walk', 'wait', 'transfers', 'gc']
rows = []
with pd.ExcelWriter(f'{OUT}/skims_area_v2.xlsx') as xw:
    pd.DataFrame({'AggCode': AREAS, 'AggAreaName': names.values, 'segment': areas['segment'].values if 'segment' in areas else ['trunk' if a in TRUNK else 'branch' for a in AREAS],
                  'on_hf_lrt_3': [a in ON_LINE for a in AREAS], 'bus_access_walk_min': accb['access'].round(2).values, 'bus_egress_walk_min': accb['egress'].round(2).values,
                  'lrt_access_walk_min': lrt_acc['access_walk_min_pop'].round(2).values, 'lrt_egress_walk_min': lrt_acc['egress_walk_min_emp'].round(2).values}).to_excel(xw, sheet_name='areas', index=False)
    for m in MODES:
        for c in COMPS + (['ivt_scheduled'] if m == 'bus' else []) + (['ivt_lrt_only', 'gateway_o', 'gateway_d', 'legs'] if m.startswith('lrt') else []) + ['status']:
            M = SK[m][c]
            if c in ('gateway_o', 'gateway_d'): M = M.map(lambda v: '' if v == '' else str(int(v)))
            M.to_csv(f'{OUT}/skim_{m}_{c}.csv', float_format='%.2f'); M.to_excel(xw, sheet_name=f'{m}_{c}'[:31])
            if c in COMPS or c == 'ivt_scheduled':
                for o in AREAS:
                    for d in AREAS: rows.append((o, d, m, c, M.loc[o, d], SK[m]['status'].loc[o, d]))
long = pd.DataFrame(rows, columns=['o', 'd', 'mode', 'component', 'value', 'status']); long.to_csv(f'{OUT}/skims_area_v2_long.csv', index=False, float_format='%.3f')
# trunk-pair summary weighted by 2022 transit trips
flows = {k: pd.read_csv(f'Output/corridor_v2/{k}_2022_area_v2.csv', index_col=0) for k in ['car', 'bus', 'rail', 'taxi', 'transit', 'total']}
for k in flows: flows[k].columns = flows[k].columns.astype(int); flows[k] = flows[k].reindex(index=AREAS, columns=AREAS).fillna(0.0)
OFF = ~np.eye(A, dtype=bool)
wT = flows['transit'].values * OFF
def wmean(M, w):
    v = M.values; ok = np.isfinite(v) & (w > 0); return float(np.sum(v[ok] * w[ok]) / w[ok].sum()) if w[ok].sum() > 0 else np.nan
STATION = [a for a in ON_LINE]; trunk_mask = np.zeros((A, A), bool); ix = [AREAS.index(a) for a in STATION]; trunk_mask[np.ix_(ix, ix)] = True; trunk_mask &= OFF
summ = []
for m in MODES:
    for scope, msk in [('all off-diagonal pairs', OFF), ('trunk pairs (10 station areas)', trunk_mask)]:
        w = wT * msk; avail = SK[m]['gc'].notna().values & msk
        summ.append({'mode': LABEL[m], 'scope': scope, 'pairs skimmed': int(avail.sum()), 'transit trips covered': round(float(wT[avail].sum()), 0),
                     'share of transit trips covered': round(float(wT[avail].sum() / wT[msk].sum()), 3),
                     **{c: round(wmean(SK[m][c], w), 1) for c in COMPS}})
summ = pd.DataFrame(summ); summ.to_csv(f'{OUT}/skims_summary_by_mode.csv', index=False)
print("Transit-trip-weighted skim components (min; GC in generalized minutes):"); print(summ.to_string(index=False))

Transit-trip-weighted skim components (min; GC in generalized minutes):
               mode                          scope  pairs skimmed  transit trips covered  share of transit trips covered  ivt  walk  wait  transfers   gc
                Car         all off-diagonal pairs            600                13778.0                           1.000 19.5   0.0   0.0        0.0 19.5
                Car trunk pairs (10 station areas)             90                 4289.0                           1.000 14.5   0.0   0.0        0.0 14.5
                Bus         all off-diagonal pairs            598                13766.0                           0.999 20.7   5.7   2.4        0.1 37.6
                Bus trunk pairs (10 station areas)             90                 4289.0                           1.000 14.3   4.5   1.7        0.1 27.4
     Metronit (BRT)         all off-diagonal pairs            278                 9305.0                           0.675 15.4   5.5   2.1        0.0 30.6
    

## 4. Observed 2022 flows against the skims — can λ be read off the cross-section?

The revealed choice is car against transit (bus + rail) on each off-diagonal area pair, taxi left out of the choice set. The transit alternative is the bus skim (the Metronit lines are among the bus services, so the bus GC already reflects them). A binary logit `P_T = 1 / (1 + exp(α + λ·(GC_bus − GC_car)))` is fitted to the pairs by maximum likelihood, each pair weighted by its 2022 trips (car + transit); the constant α absorbs the flat fare and everything else that is not in the skims. A second fit adds a constant per centroid-distance band, so that λ is identified from the variation *within* bands rather than from the long-versus-short contrast.

In [6]:
dGC = (SK['bus']['gc'] - SK['car']['gc']).values
T_car, T_tr = flows['car'].values * OFF, flows['transit'].values * OFF
ac = pd.read_csv('Output/lrt_v2/lrt_area_representative_station.csv', index_col=0).reindex(AREAS)
DIST_KM = np.hypot(ac['cx'].values[:, None] - ac['cx'].values[None, :], ac['cy'].values[:, None] - ac['cy'].values[None, :]) / 1000
ok = np.isfinite(dGC) & ((T_car + T_tr) > 0)
x, nc, nt, dk = dGC[ok], T_car[ok], T_tr[ok], DIST_KM[ok]
BANDS = [0, 3, 6, 10, 20, 100]; band = np.digitize(dk, BANDS[1:-1])
def loglik(u): pT = np.clip(1 / (1 + np.exp(u)), 1e-9, 1 - 1e-9); return np.sum(nt * np.log(pT) + nc * np.log(1 - pT))
fit = minimize(lambda p: -loglik(p[0] + p[1] * x), x0=[1.0, 0.05], method='Nelder-Mead', options={'xatol': 1e-7, 'fatol': 1e-7})
fitb = minimize(lambda p: -loglik(p[band] + p[-1] * x), x0=[1.0] * (len(BANDS) - 1) + [0.05], method='Nelder-Mead', options={'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 20000})
p0 = nt.sum() / (nt + nc).sum(); ll0 = np.sum(nt * np.log(p0) + nc * np.log(1 - p0))
alpha, lam_fit = fit.x; lam_fit_band = fitb.x[-1]
# curvature-based standard error of λ (observed information of the trip-weighted likelihood, central differences)
def loglik_of(f, p): return loglik(p[0] + p[1] * x) if len(p) == 2 else loglik(p[band] + p[-1] * x)
def se_lam(f, k, h=1e-3):
    pp, pm = f.x.copy(), f.x.copy(); pp[k] += h; pm[k] -= h
    d2 = (loglik_of(f, pp) - 2 * loglik_of(f, f.x) + loglik_of(f, pm)) / h ** 2
    return float(1 / np.sqrt(-d2)) if d2 < 0 else np.nan
se1, seb = se_lam(fit, 1), se_lam(fitb, len(fitb.x) - 1)
calib = pd.DataFrame([{'fit': 'constant + λ', 'pairs': int(ok.sum()), 'trips': round(float((nc + nt).sum())), 'lambda (per gen-min)': lam_fit, 'se': se1, 'rho2 vs constant-only': 1 - (-fit.fun) / ll0, 'observed transit share': p0},
                      {'fit': 'distance-band constants + λ', 'pairs': int(ok.sum()), 'trips': round(float((nc + nt).sum())), 'lambda (per gen-min)': lam_fit_band, 'se': seb, 'rho2 vs constant-only': 1 - (-fitb.fun) / ll0, 'observed transit share': p0}])
calib.to_csv(f'{OUT}/logit_calibration_car_vs_transit.csv', index=False, float_format='%.4f')
print(calib.round(4).to_string(index=False))
# binned view, trip-weighted: by cost difference, and by distance band
bins = [-10, 0, 5, 10, 15, 20, 30, 45, 90]; b = pd.cut(x, bins)
binned = pd.DataFrame({'bin': b, 'car': nc, 'transit': nt}).groupby('bin', observed=True).agg(pairs=('car', 'size'), car=('car', 'sum'), transit=('transit', 'sum'))
binned['transit share'] = binned['transit'] / (binned['car'] + binned['transit']); binned['mid'] = [i.mid for i in binned.index]
bybd = pd.DataFrame({'band_km': pd.cut(dk, BANDS, right=False), 'car': nc, 'transit': nt, 'dGC_w': x * (nc + nt)}).groupby('band_km', observed=True).agg(pairs=('car', 'size'), car=('car', 'sum'), transit=('transit', 'sum'), dGC_w=('dGC_w', 'sum'))
bybd['transit share'] = bybd['transit'] / (bybd['car'] + bybd['transit']); bybd['mean GC bus − car'] = bybd['dGC_w'] / (bybd['car'] + bybd['transit']); bybd = bybd.drop(columns='dGC_w')
binned.round(4).to_csv(f'{OUT}/logit_calibration_binned.csv'); bybd.round(4).to_csv(f'{OUT}/logit_calibration_by_distance_band.csv')
print(); print('by cost-difference band:'); print(binned.round(3).to_string()); print(); print('by centroid-distance band:'); print(bybd.round(3).to_string())
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.scatter(x, nt / (nt + nc), s=np.clip((nc + nt) / 15, 4, 300), color=BLUE, alpha=0.35, lw=0, label='area pair (size = 2022 trips)')
ax.plot(binned['mid'], binned['transit share'], 'o-', color=ORANGE, label='observed, by cost-difference band (trip-weighted)')
xx = np.linspace(-10, 90, 200); ax.plot(xx, 1 / (1 + np.exp(alpha + lam_fit * xx)), color=INK, lw=1.5, label=f'fitted logit: λ = {lam_fit:+.4f} per generalized minute (wrong sign)')
ax.set_xlabel('GC bus − GC car (generalized minutes; fare a constant, out of the comparison)'); ax.set_ylabel('transit share (bus + rail) of car + transit, 2022 AM'); ax.set_xlim(-10, 90); ax.set_ylim(0, 1)
ax.set_title('Observed 2022 mode split against the skim cost difference, V2 area pairs'); ax.legend(frameon=False, fontsize=8)
for s_ in ('top', 'right'): ax.spines[s_].set_visible(False)
ax.grid(axis='y', color=GRID, lw=0.6); plt.tight_layout(); plt.savefig('Output/figures/skims_logit_car_vs_transit.png', dpi=150); plt.show()

                        fit  pairs  trips  lambda (per gen-min)     se  rho2 vs constant-only  observed transit share
               constant + λ    566  69818               -0.0088 0.0004                 0.0018                  0.1972
distance-band constants + λ    566  69818                0.0055 0.0004                 0.0246                  0.1972

by cost-difference band:
          pairs        car   transit  transit share   mid
bin                                                      
(-10, 0]      5   1749.019    29.780          0.017  -5.0
(0, 5]       24   4614.543  1439.105          0.238   2.5
(5, 10]      48  12168.196  3337.739          0.215   7.5
(10, 15]     57   8901.699  1676.468          0.158  12.5
(15, 20]     74   8413.506  1965.809          0.189  17.5
(20, 30]    152  13292.244  3125.861          0.190  25.0
(30, 45]    159   6056.654  1595.834          0.209  37.5
(45, 90]     47    855.343   595.899          0.411  67.5

by centroid-distance band:
           p

**λ has to be assumed.** The cross-section gives a *positive* relation between the transit share and the transit cost handicap (the fit returns a negative λ): the pairs where the bus is dearest relative to the car — long, cross-city, with a walk and a wait against a door-to-door drive — are also the pairs whose travellers are least car-available (captive bus riders, students, the low-income and Arab localities of the northern branches). Car availability is not in the skims — there is no socio-economic segmentation at the area level — and it dominates the cross-sectional variation, so the revealed 2022 split cannot identify the cost sensitivity — the plan's own caveat. The capture below therefore runs on an **assumed** λ = 0.030 per generalized minute (the range 0.02–0.05 spans the usual AM-commute values of Israeli and European urban models; in-vehicle-time coefficients of −0.02 to −0.04 per minute), with λ_T = 2λ within the transit nest, and the observed share entering only as the pivot. Fitting λ properly needs a segmented model (car availability, purpose) on the survey's person-level records — an item for the task list.

**The LRT as a premium mode.** Rail-based modes are consistently valued above the bus at equal times — reliability, ride quality, legibility of the line — and mode-choice models carry this as a rail bonus, typically 5–10 minutes of in-vehicle-time equivalent. The capture below subtracts an **LRT premium of 5 generalized minutes** (central; 0 and 10 as the bounds) from the LRT's generalized cost in the transit nest. It is an assumption of the same standing as λ, and it applies to every LRT path, feeder or direct.

In [7]:
LRT_PREMIUM = 5.0                                    # generalized minutes, rail bonus (assumption of 22 Sep 2026)
# (λ, λ_T, LRT premium) per case
LAM_CASES = {'central (λ 0.03, λ_T 0.06, premium 5)': (0.03, 0.06, 5.0), 'low λ (0.02 / 0.03, premium 5)': (0.02, 0.03, 5.0), 'high λ (0.05 / 0.10, premium 5)': (0.05, 0.10, 5.0),
             'no premium (λ central, premium 0)': (0.03, 0.06, 0.0), 'premium 10 (λ central)': (0.03, 0.06, 10.0)}
CENTRAL = 'central (λ 0.03, λ_T 0.06, premium 5)'

## 5. The LRT scenarios — incremental logit pivoted on the observed flows

Within the transit nest the LRT share is `P_LRT|T = 1 / (1 + exp(λ_T · (GC_LRT − premium − GC_bus)))`; the nest's composite cost improves by `Δ = −(1/λ_T) · ln(1 + exp(−λ_T · (GC_LRT − premium − GC_bus)))` and the car → transit shift follows the incremental logit `S' = S·e^{−λΔ} / (S·e^{−λΔ} + 1 − S)` on the pair's 2022 transit share `S` (Empirical-Bayes-smoothed towards the study-area share on thin pairs, k = 20 trips); the growth factor `S'/S` is applied to the pair's **observed** transit trips, so with the LRT removed the model returns the 2022 flows exactly. Within the nest both the existing and the new transit trips split by `P_LRT|T`: LRT trips *from bus* are `P_LRT|T` × the observed transit trips, LRT trips *from car* are `P_LRT|T` × the induced transit trips, and the bus keeps the rest. Taxi trips are carried unchanged.

In [8]:
K_EB = 20.0
n_pair = T_car + T_tr
S_raw = np.where(n_pair > 0, T_tr / np.where(n_pair > 0, n_pair, 1), np.nan)
S_piv = np.where(n_pair > 0, (n_pair * np.nan_to_num(S_raw) + K_EB * p0) / (n_pair + K_EB), np.nan)
RES = {}; scen_rows = []
for sc in LRT_SCEN:
    for lab, (lam, lt, prem) in LAM_CASES.items():
        dL = (SK[sc]['gc'] - prem - SK['bus']['gc']).values
        avail = np.isfinite(dL) & OFF & (n_pair > 0)
        P_L = np.where(avail, 1 / (1 + np.exp(lt * np.nan_to_num(dL))), 0.0)
        delta = np.where(avail, -(1 / lt) * np.log1p(np.exp(-lt * np.nan_to_num(dL))), 0.0)
        S_new = np.where(avail, S_piv * np.exp(-lam * delta) / (S_piv * np.exp(-lam * delta) + 1 - S_piv), S_piv)
        growth = np.where(avail, np.nan_to_num(S_new) / np.where(np.nan_to_num(S_piv) > 0, S_piv, 1), 1.0)
        T_tr_new = np.minimum(T_tr * growth, n_pair); T_car_new = n_pair - T_tr_new
        T_lrt = T_tr_new * P_L; T_bus_new = T_tr_new - T_lrt
        from_bus = T_tr * P_L; from_car = T_lrt - from_bus
        RES[(sc, lab)] = {'P_LRT|T': P_L, 'T_lrt': T_lrt, 'T_bus': T_bus_new, 'T_car': T_car_new, 'from_car': from_car, 'from_bus': from_bus}
        scen_rows.append({'scenario': LABEL[sc], 'case': lab, 'λ': lam, 'λ_T': lt, 'LRT premium': prem, 'LRT trips 06–09': round(float(T_lrt.sum())), 'of which from bus': round(float(from_bus.sum())), 'of which from car': round(float(from_car.sum())),
                          'bus trips after': round(float(T_bus_new.sum())), 'car trips after': round(float(T_car_new.sum())), 'transit share before': round(float(T_tr.sum() / n_pair.sum()), 4), 'transit share after': round(float(T_tr_new.sum() / n_pair.sum()), 4),
                          'LRT share of transit': round(float(T_lrt.sum() / T_tr_new.sum()), 3), 'LRT trips on trunk pairs': round(float(T_lrt[trunk_mask].sum())), 'trunk pairs P_LRT|T (trip-wtd)': round(float(np.average(P_L[trunk_mask], weights=np.maximum(T_tr[trunk_mask], 1e-9))), 3)})
scen = pd.DataFrame(scen_rows); scen.to_csv(f'{OUT}/lrt_capture_scenarios.csv', index=False)
print(f"2022 AM off-diagonal trips in the 25 areas: car {T_car.sum():,.0f}, transit {T_tr.sum():,.0f} (share {T_tr.sum() / n_pair.sum():.3f}), taxi {(flows['taxi'].values * OFF).sum():,.0f} (unchanged)")
print(scen.to_string(index=False))
for sc in LRT_SCEN:
    r = RES[(sc, CENTRAL)]
    pd.DataFrame(r['T_lrt'], index=AREAS, columns=AREAS).to_csv(f'{OUT}/lrt_trips_2022_{sc}_central.csv', float_format='%.1f')
    pd.DataFrame(r['P_LRT|T'], index=AREAS, columns=AREAS).to_csv(f'{OUT}/lrt_share_of_transit_{sc}_central.csv', float_format='%.3f')
pd.DataFrame(S_piv, index=AREAS, columns=AREAS).to_csv(f'{OUT}/transit_share_2022_pivot.csv', float_format='%.4f')

2022 AM off-diagonal trips in the 25 areas: car 56,445, transit 13,778 (share 0.196), taxi 720 (unchanged)
           scenario                                  case    λ  λ_T  LRT premium  LRT trips 06–09  of which from bus  of which from car  bus trips after  car trips after  transit share before  transit share after  LRT share of transit  LRT trips on trunk pairs  trunk pairs P_LRT|T (trip-wtd)
LRT all underground central (λ 0.03, λ_T 0.06, premium 5) 0.03 0.06          5.0             4114               3751                363            10749            55360                0.1962               0.2117                 0.277                      1619                           0.337
LRT all underground        low λ (0.02 / 0.03, premium 5) 0.02 0.03          5.0             5812               4985                828            10175            54235                0.1962               0.2277                 0.364                      2143                           0.415
LRT all underg

## 6. Flows on the trunk links — today's bus movements against the LRT loads

Every LRT trip (central case: λ 0.03, λ_T 0.06, premium 5) is loaded on the trunk between its gateway station areas (the pair itself when both ends are on the line), link by link along the `Order_T1` sequence 201 → 210, in its direction (up = away from Tirat Carmel). Today's bus, transit and car potential movements on the same links come from step 24's tree-network profile (`corridor_v2_network_link_flows.csv`) — the trips that could traverse the link if every such trip used the corridor, so the two are on the same footing: corridor-internal, 06:00–09:00, and no route choice against parallel services. The peak-hour LRT load uses step 27's network transit factor.

In [9]:
lf = pd.read_csv('Output/corridor_v2/corridor_v2_network_link_flows.csv')
trunk_links = lf[(lf['segment'] == 'trunk')][['link', 'from_code', 'to_code', 'from_area', 'to_area']].drop_duplicates().sort_values('link')
trunk_links = trunk_links[trunk_links['to_code'].isin(ON_LINE) & trunk_links['from_code'].isin(ON_LINE)].reset_index(drop=True)
pos = {a: i for i, a in enumerate(ON_LINE)}
phf = pd.read_csv('Output/corridor_v2/peak_hour_factors_v2_applied.csv'); phf_net = phf[(phf['scope'] == 'network') & (phf['layer'] == 'bus')].drop_duplicates('direction').set_index('direction')['PHF3h applied']
PHF_T = {d: float(phf_net[d]) for d in ('up', 'down')}
def load_links(T, g_o, g_d):
    L = {(l, d): 0.0 for l in trunk_links['link'] for d in ('up', 'down')}
    for i, o in enumerate(AREAS):
        for j, d in enumerate(AREAS):
            t = T[i, j]
            if t <= 0 or g_o.loc[o, d] == '': continue
            a, b = pos[g_o.loc[o, d]], pos[g_d.loc[o, d]]
            if a == b: continue
            dirn = 'up' if b > a else 'down'
            for k in range(min(a, b), max(a, b)):
                L[(trunk_links.loc[k, 'link'], dirn)] += t
    return L
prof_rows = []
for _, r in trunk_links.iterrows():
    for d in ('up', 'down'):
        base = lf[(lf['link'] == r['link']) & (lf['direction'] == d)].set_index('layer')['flow_3h']
        basep = lf[(lf['link'] == r['link']) & (lf['direction'] == d)].set_index('layer')['flow_peak_hour']
        prof_rows.append({'link': r['link'], 'from_area': r['from_area'], 'to_area': r['to_area'], 'direction': d, 'bus_2022_3h': base.get('bus', np.nan), 'transit_2022_3h': base.get('transit', np.nan), 'car_2022_3h': base.get('car', np.nan),
                          'bus_2022_peak_hour': basep.get('bus', np.nan), 'car_2022_peak_hour': basep.get('car', np.nan)})
prof = pd.DataFrame(prof_rows)
for sc, tag in (('lrt_all_underground', 'lrt_ug'), ('lrt_all_ground', 'lrt_ground'), ('lrt_design_50kmh', 'lrt_d50')):
    for lab, short in ((CENTRAL, ''), ('low λ (0.02 / 0.03, premium 5)', '_low'), ('high λ (0.05 / 0.10, premium 5)', '_high'), ('no premium (λ central, premium 0)', '_prem0'), ('premium 10 (λ central)', '_prem10')):
        L = load_links(RES[(sc, lab)]['T_lrt'], SK[sc]['gateway_o'], SK[sc]['gateway_d'])
        prof[f'{tag}{short}_3h'] = [L[(l, d)] for l, d in zip(prof['link'], prof['direction'])]
        if short == '': prof[f'{tag}_peak_hour'] = prof[f'{tag}_3h'] * prof['direction'].map(PHF_T)
prof['lrt_ug_share_of_bus_2022'] = prof['lrt_ug_3h'] / prof['bus_2022_3h']
prof = prof.round(1); prof.to_csv(f'{OUT}/trunk_link_flows_bus_vs_lrt.csv', index=False)
print(prof[['link', 'from_area', 'to_area', 'direction', 'bus_2022_3h', 'car_2022_3h', 'lrt_ug_3h', 'lrt_ground_3h', 'lrt_d50_3h', 'lrt_ug_low_3h', 'lrt_ug_high_3h', 'lrt_ug_prem0_3h', 'lrt_ug_prem10_3h', 'lrt_ug_peak_hour', 'lrt_ground_peak_hour']].to_string(index=False))
# boardings by area (origin gateway) for the central case, both scenarios
bo = []
for sc in LRT_SCEN:
    T = RES[(sc, CENTRAL)]['T_lrt']; g_o = SK[sc]['gateway_o']
    s = pd.Series(0.0, index=ON_LINE)
    for i, o in enumerate(AREAS):
        for j, d in enumerate(AREAS):
            if T[i, j] > 0 and g_o.loc[o, d] != '': s[g_o.loc[o, d]] += T[i, j]
    bo.append(s.rename(LABEL[sc]))
board = pd.concat(bo, axis=1); board.insert(0, 'area', names.reindex(ON_LINE).values); board.round(0).to_csv(f'{OUT}/lrt_boardings_by_station_area_central.csv')
# figure: link profile up direction, bus today vs LRT scenarios
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for ax, d in zip(axes, ('up', 'down')):
    p = prof[prof['direction'] == d]; xs = np.arange(len(p))
    ax.bar(xs - 0.33, p['bus_2022_3h'], 0.21, color=MUTED, label='bus today (2022 potential movements)')
    ax.bar(xs - 0.11, p['lrt_ug_3h'], 0.21, color=BLUE, label='LRT all underground (central)')
    ax.bar(xs + 0.11, p['lrt_ground_3h'], 0.21, color=ORANGE, label='LRT all ground (central)')
    ax.bar(xs + 0.33, p['lrt_d50_3h'], 0.21, color=AQUA, label='LRT design 50 km/h + 10 s (central)')
    ax.vlines(xs - 0.11, p['lrt_ug_low_3h'], p['lrt_ug_high_3h'], color=INK, lw=1); ax.vlines(xs + 0.11, p['lrt_ground_low_3h'], p['lrt_ground_high_3h'], color=INK, lw=1); ax.vlines(xs + 0.33, p['lrt_d50_low_3h'], p['lrt_d50_high_3h'], color=INK, lw=1)
    ax.set_ylabel(f'trips 06:00–09:00, {d}'); ax.grid(axis='y', color=GRID, lw=0.6)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
axes[1].set_xticks(np.arange(len(p))); axes[1].set_xticklabels([f"{a}→{b}" for a, b in zip(p['from_area'], p['to_area'])], rotation=35, ha='right', fontsize=8)
axes[0].legend(frameon=False, fontsize=8); axes[0].set_title('Trunk links: today\'s bus movements against the LRT loads (whiskers: low … high λ, premium 5)')
plt.tight_layout(); plt.savefig('Output/figures/skims_trunk_link_flows_bus_vs_lrt.png', dpi=150); plt.show()

 link              from_area                to_area direction  bus_2022_3h  car_2022_3h  lrt_ug_3h  lrt_ground_3h  lrt_d50_3h  lrt_ug_low_3h  lrt_ug_high_3h  lrt_ug_prem0_3h  lrt_ug_prem10_3h  lrt_ug_peak_hour  lrt_ground_peak_hour
    1            TiratCarmel        Matam-NeotPeres        up        828.6       4012.7      393.0          243.4       512.2          470.0           335.6            318.7             477.7             215.9                 133.7
    1            TiratCarmel        Matam-NeotPeres      down        414.9       1192.2      190.4          133.1       232.5          231.0           162.9            153.8             232.2              87.0                  60.8
    2        Matam-NeotPeres    HofCarmel-NeveDavid        up        826.9       2621.5      325.7          176.4       435.4          402.0           263.2            262.6             397.8             179.0                  96.9
    2        Matam-NeotPeres    HofCarmel-NeveDavid      down       1622

## 7. Where the LRT trips come from — the pair-level comparison

In [10]:
r = RES[('lrt_all_underground', CENTRAL)]; r2 = RES[('lrt_all_ground', CENTRAL)]; r3 = RES[('lrt_design_50kmh', CENTRAL)]
pairs = pd.DataFrame({'o': np.repeat(AREAS, A), 'd': np.tile(AREAS, A)})
pairs['o_name'] = names.reindex(pairs['o']).values; pairs['d_name'] = names.reindex(pairs['d']).values
for k, M in [('car_2022', T_car), ('transit_2022', T_tr), ('gc_car', SK['car']['gc'].values), ('gc_bus', SK['bus']['gc'].values), ('gc_brt', SK['brt']['gc'].values),
             ('gc_lrt_ug', SK['lrt_all_underground']['gc'].values), ('gc_lrt_ground', SK['lrt_all_ground']['gc'].values), ('gc_lrt_ug_with_premium', SK['lrt_all_underground']['gc'].values - LRT_PREMIUM),
             ('P_lrt_of_transit_ug', r['P_LRT|T']), ('lrt_trips_ug', r['T_lrt']), ('lrt_from_bus_ug', r['from_bus']), ('lrt_from_car_ug', r['from_car']),
             ('P_lrt_of_transit_ground', r2['P_LRT|T']), ('lrt_trips_ground', r2['T_lrt']), ('gc_lrt_d50', SK['lrt_design_50kmh']['gc'].values), ('P_lrt_of_transit_d50', r3['P_LRT|T']), ('lrt_trips_d50', r3['T_lrt'])]:
    pairs[k] = M.ravel()
pairs['lrt_legs_ug'] = SK['lrt_all_underground']['legs'].values.ravel(); pairs['lrt_status_ug'] = SK['lrt_all_underground']['status'].values.ravel()
pairs = pairs[pairs['o'] != pairs['d']].round(2); pairs.to_csv(f'{OUT}/pair_flows_and_skims_2022.csv', index=False)
top = pairs.sort_values('lrt_trips_ug', ascending=False).head(15)[['o_name', 'd_name', 'car_2022', 'transit_2022', 'gc_car', 'gc_bus', 'gc_lrt_ug', 'gc_lrt_ground', 'P_lrt_of_transit_ug', 'lrt_trips_ug', 'lrt_trips_ground', 'lrt_legs_ug']]
print('Fifteen largest LRT flows (central case, all underground):'); print(top.to_string(index=False))
by_legs = pairs.groupby('lrt_legs_ug').agg(pairs=('o', 'size'), transit_2022=('transit_2022', 'sum'), lrt_ug=('lrt_trips_ug', 'sum'), lrt_ground=('lrt_trips_ground', 'sum'), lrt_d50=('lrt_trips_d50', 'sum')).round(0)
by_legs['P_ug'] = (by_legs['lrt_ug'] / by_legs['transit_2022']).round(3); print(); print(by_legs.to_string())
by_legs.to_csv(f'{OUT}/lrt_trips_by_path_type.csv')

Fifteen largest LRT flows (central case, all underground):
                o_name                 d_name  car_2022  transit_2022  gc_car  gc_bus  gc_lrt_ug  gc_lrt_ground  P_lrt_of_transit_ug  lrt_trips_ug  lrt_trips_ground lrt_legs_ug
              Nazareth             KiryatHaim      0.00        253.27   53.46  105.74     111.12         114.52                 0.49        125.20            112.35 bus→LRT→brt
BatGalim-KiryatEliezer        Matam-NeotPeres    356.52        265.12   17.68   27.41      42.17          51.92                 0.36        107.15             67.72   directLRT
   KiryatBialik Center              LowerCity     78.55        195.99   22.85   40.75      46.85          52.65                 0.48        104.20             84.06     brt→LRT
           TiratCarmel        Matam-NeotPeres   1924.92        211.66   12.27   35.32      48.34          53.04                 0.38        100.11             79.80   directLRT
      Hecht-Shprintzak        Matam-NeotPeres    146.13 

## Findings

- **The skim set.** Five complete 25 × 25 matrices per component in `Output/skims/` (`skims_area_v2.xlsx`, one sheet per mode × component, and the same as CSV). Car is full (survey door-to-door, 35 measured / 590 derived cells); bus is 422 derived (observed running time of the direct services, step 30) and 178 assumed (scaled fastest-path floor with one transfer counted); Metronit is a direct-service skim on 278 pairs and *not available* elsewhere; the two LRT scenarios are full — 90 direct pairs on the ten station areas and 510 through a feeder leg to the cheapest gateway station area. With the LRT–Metronit transfer free, the **Metronit is the feeder at one or both ends on 378 of those 510 pairs** (underground) and the bus on the rest; 75 / 119 cells rest on an assumed bus leg. Money is out of the comparison by decision (flat integrated fare with a daily cap: the same for bus, Metronit and LRT and for every pair). Transit-trip-weighted on the 90 trunk pairs: car 14.7, bus 27.8 (IVT 14.9, walk 4.8, wait 1.4), Metronit 25.6 on its 72 pairs, LRT 44.0 underground (IVT 12.6, walk 13.2, wait 2.5) / 51.8 ground (IVT 20.4). Over all 600 pairs: car 20.2, bus 38.5, LRT 64.6 / 70.0 — the off-line pairs carry a 15–30-minute feeder ride.
- **On physical cost the LRT is dearer than the bus on 560 of the 598 pairs with a bus skim** (on the direct trunk pairs by 4–28 generalized minutes, median 15); with the 5-minute premium it is cheaper on 96 pairs carrying 441 transit trips. What makes it dearer on the trunk is station access (13.2 against 4.8 min of walking, weighted twice); off the line it is the feeder ride, no longer the transfer where the Metronit feeds it.
- **The 2022 flows do not calibrate λ.** On 573 pairs and 67,700 car + transit trips the binary logit of the transit share on `GC_bus − GC_car` returns λ = −0.011 per generalized minute (wrong sign, ρ² 0.003); with a constant per distance band, λ = +0.0002 (ρ² 0.036) — no sensitivity within bands either. The transit share is 0.11 under 3 km, 0.25 at 3–6 km, 0.16–0.17 at 6–20 km and 0.41 beyond 20 km (Nazareth, Shefaram), while the bus's cost handicap grows steadily from 11 to 28 generalized minutes over the same bands: the pairs where the bus is worst placed are those whose travellers are least car-available, and car availability is not in the skims. The revealed split therefore identifies car availability, not the cost sensitivity — the plan's own caveat, now shown on the data. λ is **assumed**: 0.03 per generalized minute (range 0.02–0.05), λ_T = 2λ within the transit nest, with the observed share entering only as the pivot; the LRT carries a **premium of 5 generalized minutes** (rail bonus; 0 and 10 as bounds).
- **What the LRT would draw, on the 2022 corridor-internal market.** Central case, all underground: **3,332 LRT trips in 06:00–09:00**, 3,037 of them from today's 11,664 bus and rail trips (26 %) and 295 induced from car (the transit share of car + transit moves from 0.171 to 0.185); all at ground level, 2,544 (21 %). Of the three assumptions the premium and the free Metronit transfer each add about a fifth: without the premium 2,658 underground, of which some 400 come from the free Metronit transfer; with a 10-minute premium 4,122. The λ range spans 2,357–4,855 underground and 1,472–4,277 at ground level — still wider than the difference between the two alignments. On the 90 trunk pairs the LRT takes 34 % of the transit trips underground and 25 % at ground level (1,580 / 1,112 trips); fed by the Metronit or a bus at one end 35 %, at the destination end by the Metronit 50 % (short egress hops into the Krayot), fed at both ends 13 % by Metronit and next to nothing by two bus legs.
- **A third LRT regime — the specified design speed (50 km/h between stops, 10 s per stop, 5-minute headway; step 25).** Its in-vehicle time on the trunk pairs is 8.0 min (against 12.6 underground and 20.4 at ground level), the partial generalized cost 39.4 (44.0 / 51.8), 61.1 over all pairs. On physical cost it is still dearer than the bus on 494 of the 598 pairs (on the direct trunk pairs by −8 to +26 generalized minutes, median 10.5); with the 5-minute premium it beats the bus on 192 pairs carrying 1,764 transit trips. Central case: **3,970 LRT trips** in 06:00–09:00 (3,534 from bus, 437 from car; 31 % of the transit trips, 40 % on the trunk pairs), λ range 3,227–5,272, premium 0 / 10 → 3,210 / 4,842; 1,934 on the trunk pairs. Trunk links 108–1,237 (peak hour 64–730), busiest Namal-Giborim → Hamifrats down 1,237; boardings Hamifrats 1,237, Namal-Giborim 888, Hecht-Shprintzak 464, Bat Galim 428. Read it as the ceiling of the alignment: the specification carries no acceleration or braking, so it sits 35 % below the calibrated underground time, and the 19 % more trips it draws than the underground case is what that ceiling is worth — less than the λ range.
- **Where it boards and what it carries.** Boardings concentrate at the two Krayot-facing ends — Hamifrats 995 and Namal-Giborim 747 underground, the gateways of the branch areas fed by the Metronit — then Hecht-Shprintzak 391 and Bat Galim 353. On the trunk links the central underground loads are 79–995 trips per link and direction over the three hours (peak hour 47–588), heaviest on Namal-Giborim → Hamifrats towards the Krayot (995), Lower City → Namal-Giborim (799) and Matam → Hof Carmel (678): a third to 45 % of the bus movements on the same links today (200–2,030), and 84 % on Namal-Giborim → Hamifrats up (637 against 759). The ground scenario carries 44–595. The largest pair flows are Bat Galim → Matam (122), Hecht-Shprintzak → Matam (116) and Tirat Carmel → Matam (104) — the Matam employment end — then Nazareth → Lower City (75) by feeder bus and Kiryat Bialik → Bat Galim (43) by Metronit feeder.
- **How to read the numbers.** They are 2022 trips with both ends inside the 25 areas — no external trips, no forecast-year growth (the 2040 / 2050 matrices of the forecast step apply the same machinery), no capacity, no route choice against the parallel bus lines that stay, money out of the comparison by decision (flat integrated fare), and three perception assumptions rather than estimates: λ, the LRT premium and the free LRT–Metronit transfer. Two things would move them most: λ itself, which needs a segmented estimation on the survey's person-level records (car availability, purpose) or a transferred value with a cited source; and station access, which the feeder assumption and the population-weighted walk treat crudely. The station plan of the branches (E1) would replace the feeder composite for 15 of the 25 areas.
- **Outputs.** `Output/skims/skims_area_v2.xlsx`, `skim_{car,bus,brt,lrt_all_underground,lrt_all_ground,lrt_design_50kmh}_{ivt,walk,wait,transfers,gc,status}.csv` (+ `bus_ivt_scheduled`, LRT `ivt_lrt_only`, `gateway_o`, `gateway_d`, `legs`), `skims_area_v2_long.csv`, `skims_summary_by_mode.csv`; `logit_calibration_car_vs_transit.csv`, `logit_calibration_binned.csv`, `logit_calibration_by_distance_band.csv`; `lrt_capture_scenarios.csv`, `lrt_trips_2022_{scenario}_central.csv`, `lrt_share_of_transit_{scenario}_central.csv`, `transit_share_2022_pivot.csv`, `pair_flows_and_skims_2022.csv`, `lrt_trips_by_path_type.csv`, `lrt_boardings_by_station_area_central.csv`, `trunk_link_flows_bus_vs_lrt.csv`; figures `skims_logit_car_vs_transit.png`, `skims_trunk_link_flows_bus_vs_lrt.png`.